In [1]:
from pathlib import Path

# Comme le notebook est dans notebooks/, on remonte d'un niveau
DATA_DIR = Path("../data")

for item in DATA_DIR.iterdir():
    print(item)

..\data\ann_txt_files
..\data\DetectOnco_Final.csv
..\data\GuideAnnotation.docx


structure du csv

In [2]:
import pandas as pd

df = pd.read_csv(DATA_DIR / "DetectOnco_Final.csv")

print("Colonnes :", df.columns.tolist())
print("Nombre de lignes :", len(df))
print("\nTypes d'entités (label) :")
print(df['label'].unique())
print("\nNombre de documents uniques :")
print(df['doc_name'].nunique())

Colonnes : ['doc_name', 'id', 'label', 'code', 'content', 'full_span']
Nombre de lignes : 71126

Types d'entités (label) :
['topographie' 'differenciation' 'morphologie' 'expression_CIM']

Nombre de documents uniques :
1300


structure du fichier ann

In [3]:
ann_dir = DATA_DIR / "ann_txt_files"
ann_files = sorted(ann_dir.glob("*.ann"))
print(f"Nombre de fichiers .ann : {len(ann_files)}")

# Juste la structure d'un fichier, ligne par ligne, avec repr() pour voir le format exact
with open(ann_files[0], "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"\nNombre de lignes dans {ann_files[0].name} :", len(lines))
print("Format de la première ligne (repr) :")
print(repr(lines[0]) if lines else "fichier vide")

Nombre de fichiers .ann : 1301

Nombre de lignes dans cc_onco1.ann : 40
Format de la première ligne (repr) :
'T1\tmorphologie 2300 2311\tadénopathie\n'


distribution des longueurs de texte

In [4]:
txt_files = sorted(ann_dir.glob("*.txt"))
print(f"Nombre de fichiers .txt : {len(txt_files)}")

lengths = []
for f in txt_files:
    content = f.read_text(encoding="utf-8")
    lengths.append(len(content))

import statistics
print(f"Longueur moyenne : {statistics.mean(lengths):.0f} caractères")
print(f"Min : {min(lengths)}, Max : {max(lengths)}")

Nombre de fichiers .txt : 1301
Longueur moyenne : 5287 caractères
Min : 1300, Max : 16321


In [5]:
with open(ann_files[0], "r", encoding="utf-8") as f:
    lines = f.readlines()

# Voir les types de préfixes de ligne (T, #, N, R...)
prefixes = [line.split('\t')[0][0] for line in lines if line.strip()]
from collections import Counter
print(Counter(prefixes))

Counter({'T': 20, '#': 20})


In [6]:
with open(ann_files[0], "r", encoding="utf-8") as f:
    lines = f.readlines()

# Isoler une ligne T et la ligne # qui lui correspond probablement
t_lines = [l for l in lines if l.startswith('T')]
hash_lines = [l for l in lines if l.startswith('#')]

print("Exemple ligne T :")
print(repr(t_lines[0]))

print("\nExemple ligne # :")
print(repr(hash_lines[0]))

Exemple ligne T :
'T1\tmorphologie 2300 2311\tadénopathie\n'

Exemple ligne # :
'#1\tICD-O T1\t8000/1\n'


détection de spans discontinus 

In [7]:
from pathlib import Path

ann_files = sorted((DATA_DIR / "ann_txt_files").glob("*.ann"))
discontinuous_count = 0

for ann_path in ann_files:
    with open(ann_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("T") and ";" in line.split("\t")[1]:
                discontinuous_count += 1
                print(f"Span discontinu trouvé : {ann_path.name}")
                break

print(f"\nTotal fichiers avec spans discontinus : {discontinuous_count}")

Span discontinu trouvé : cc_onco1.ann
Span discontinu trouvé : cc_onco1000.ann
Span discontinu trouvé : cc_onco1006.ann
Span discontinu trouvé : cc_onco1008.ann
Span discontinu trouvé : cc_onco1009.ann
Span discontinu trouvé : cc_onco1010.ann
Span discontinu trouvé : cc_onco1011.ann
Span discontinu trouvé : cc_onco1016.ann
Span discontinu trouvé : cc_onco1019.ann
Span discontinu trouvé : cc_onco102.ann
Span discontinu trouvé : cc_onco1020.ann
Span discontinu trouvé : cc_onco1021.ann
Span discontinu trouvé : cc_onco1022.ann
Span discontinu trouvé : cc_onco1023.ann
Span discontinu trouvé : cc_onco1025.ann
Span discontinu trouvé : cc_onco1027.ann
Span discontinu trouvé : cc_onco103.ann
Span discontinu trouvé : cc_onco1031.ann
Span discontinu trouvé : cc_onco1038.ann
Span discontinu trouvé : cc_onco104.ann
Span discontinu trouvé : cc_onco1041.ann
Span discontinu trouvé : cc_onco1043.ann
Span discontinu trouvé : cc_onco1046.ann
Span discontinu trouvé : cc_onco1049.ann
Span discontinu trouvé

In [8]:
for ann_path in ann_files:
    with open(ann_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("T") and ";" in line.split("\t")[1]:
                print(ann_path.name)
                print(repr(line))
                break
        else:
            continue
        break

cc_onco1.ann
'T13\tmorphologie 3225 3234;3246 3258\tcarcinome microcytaire\n'
